# OPTUNA — Optimización inteligente con Walk-Forward

## ¿Qué es Walk-Forward?
La técnica más importante para evitar sobreoptimización:

```
Año 1-3: TRAIN (optimizar) → Año 4: TEST (validar) → ¿funciona fuera de muestra?
Año 2-4: TRAIN             → Año 5: TEST
Año 3-5: TRAIN             → Año 6: TEST
```

Si la estrategia funciona en TODOS los periodos de test → es robusta, no curva ajustada.

## Configuración recomendada:
- **Train window:** 3 años
- **Test window:** 1 año
- **Mínimo de operaciones:** 20 por periodo

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import yfinance as yf
import optuna
import vectorbt as vbt
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── CONFIGURACION ──────────────────────────────────────────
SIMBOLO      = 'SPY'
INICIO       = '2015-01-01'
FIN          = '2026-05-01'
TRAIN_YEARS  = 3      # años de entrenamiento
TEST_YEARS   = 1      # años de validacion
N_TRIALS     = 50     # combinaciones por ventana (mas = mas lento pero mas preciso)
MIN_OPS      = 20     # minimo de operaciones por periodo

print('Descargando datos...')
data  = yf.download(SIMBOLO, start=INICIO, end=FIN, progress=False, auto_adjust=True)
close = data['Close'].squeeze()
close.index = pd.to_datetime(close.index).tz_localize(None)
print(f'OK: {len(close)} dias | {close.index[0].date()} - {close.index[-1].date()}')

In [ ]:
# ── FUNCION DE BACKTEST ────────────────────────────────────
def backtest_simple(serie, ema_r, ema_l, adx_th=20):
    """Backtest EMA crossover con filtro ADX sobre una serie de precios"""
    fast = serie.ewm(span=ema_r, adjust=False).mean()
    slow = serie.ewm(span=ema_l, adjust=False).mean()
    
    # ADX simplificado
    diff = abs(fast - slow) / slow * 100
    adx_proxy = diff.rolling(14).mean()
    
    entries = (fast > slow) & (fast.shift() <= slow.shift()) & (adx_proxy > adx_th/10)
    exits   = (fast < slow) & (fast.shift() >= slow.shift())
    
    pf = vbt.Portfolio.from_signals(
        serie, entries.fillna(False), exits.fillna(False),
        init_cash=10000, fees=0.001, freq='1D'
    )
    n = pf.trades.count()
    if n < MIN_OPS:
        return None, n
    return pf.total_return(), n

# ── WALK-FORWARD ──────────────────────────────────────────
def walk_forward():
    resultados = []
    inicio = close.index[0]
    fin    = close.index[-1]
    
    cursor = inicio + pd.DateOffset(years=TRAIN_YEARS)
    ventana = 1
    
    while cursor + pd.DateOffset(years=TEST_YEARS) <= fin:
        train_start = cursor - pd.DateOffset(years=TRAIN_YEARS)
        train_end   = cursor
        test_start  = cursor
        test_end    = cursor + pd.DateOffset(years=TEST_YEARS)
        
        train = close[train_start:train_end]
        test  = close[test_start:test_end]
        
        # Optimizar en TRAIN
        def objetivo(trial):
            ema_r  = trial.suggest_int('ema_r', 5, 20)
            ema_l  = trial.suggest_int('ema_l', 20, 50)
            adx_th = trial.suggest_int('adx_th', 10, 30)
            if ema_r >= ema_l: return -1.0
            ret, n = backtest_simple(train, ema_r, ema_l, adx_th)
            return float(ret) if ret is not None else -1.0
        
        study = optuna.create_study(direction='maximize',
                                    sampler=optuna.samplers.TPESampler(seed=42))
        study.optimize(objetivo, n_trials=N_TRIALS)
        p = study.best_params
        
        # Validar en TEST con los mejores parametros
        ret_test, n_test = backtest_simple(test, p['ema_r'], p['ema_l'], p['adx_th'])
        ret_val = float(ret_test) * 100 if ret_test is not None else None
        
        resultados.append({
            'ventana'   : ventana,
            'train'     : f"{train_start.year}-{train_end.year}",
            'test'      : f"{test_start.year}-{test_end.year}",
            'ema_r'     : p['ema_r'],
            'ema_l'     : p['ema_l'],
            'adx_th'    : p['adx_th'],
            'ret_test%' : ret_val,
            'ops_test'  : n_test
        })
        
        print(f'Ventana {ventana}: train {train_start.year}-{train_end.year} | '
              f'test {test_start.year}-{test_end.year} | '
              f'EMA {p["ema_r"]}/{p["ema_l"]} ADX>{p["adx_th"]} | '
              f'Ret test: {ret_val:.1f}% ({n_test} ops)' if ret_val else 
              f'Ventana {ventana}: pocas operaciones en test')
        
        cursor += pd.DateOffset(years=TEST_YEARS)
        ventana += 1
    
    return pd.DataFrame(resultados)

print('Ejecutando Walk-Forward Analysis...')
print('(puede tardar 2-3 minutos)\n')
df_wf = walk_forward()

In [ ]:
# ── RESULTADOS ────────────────────────────────────────────
print('\n' + '='*60)
print('  RESULTADOS WALK-FORWARD')
print('='*60)
print(df_wf.to_string(index=False))

validos = df_wf[df_wf['ret_test%'].notna()]
positivos = (validos['ret_test%'] > 0).sum()
total_v   = len(validos)

print(f'\nPeriodos positivos fuera de muestra: {positivos}/{total_v}')
if total_v > 0:
    pct_ok = positivos / total_v * 100
    print(f'Porcentaje de exito out-of-sample: {pct_ok:.0f}%')
    if pct_ok >= 70:
        print('RESULTADO: Estrategia ROBUSTA (>70% periodos positivos)')
    elif pct_ok >= 50:
        print('RESULTADO: Estrategia ACEPTABLE (50-70% periodos positivos)')
    else:
        print('RESULTADO: Estrategia FRAGIL (<50% periodos positivos — riesgo de overfitting)')

print('\nParametros mas frecuentes (los mas robustos):')
print(f'  EMA rapida mas comun : {validos["ema_r"].mode().iloc[0] if len(validos) > 0 else "N/A"}')
print(f'  EMA lenta mas comun  : {validos["ema_l"].mode().iloc[0] if len(validos) > 0 else "N/A"}')
print(f'  ADX umbral mas comun : {validos["adx_th"].mode().iloc[0] if len(validos) > 0 else "N/A"}')